In [ ]:
"""
Script 4: E-Prime vs Neural Stimulation Timing Alignment Visualization
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
import os, warnings
warnings.filterwarnings('ignore')

PLOTS_DIR = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

events = pd.read_csv('C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data\\events_data_processed.csv')
lfp    = pd.read_csv('C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data\\neural_lfp_power_processed.csv')

# ── Key timing numbers ────────────────────────────────────────────────────────
eprime_start_ms = events['fixation_onset_ms'].min()        # 157381
eprime_end_ms   = events['feedback_offset_ms'].max()       # 342014
neural_start_ms = lfp['eprime_time_ms'].min()              # 349000
neural_end_ms   = lfp['eprime_time_ms'].max()              # 628500

# Identify real spikes: 0 -> non-zero -> 0 cycles
lfp_s = lfp.sort_values('eprime_time_ms').reset_index(drop=True)
lfp_s['prev_ma'] = lfp_s['stim_ma_left'].shift(1).fillna(0)

spike_starts = lfp_s[(lfp_s['prev_ma'] == 0) & (lfp_s['stim_ma_left'] > 0)].copy()
spike_ends   = lfp_s[(lfp_s['prev_ma'] > 0)  & (lfp_s['stim_ma_left'] == 0)].copy()

first_spike_start_ms = spike_starts['eprime_time_ms'].iloc[0]   # 365500
first_spike_end_ms   = spike_ends['eprime_time_ms'].iloc[0]     # 368000
last_spike_start_ms  = spike_starts['eprime_time_ms'].iloc[-1]  # 622000
last_spike_end_ms    = spike_ends['eprime_time_ms'].iloc[-1]    # 624000

gap_task_to_neural_ms   = neural_start_ms - eprime_end_ms        # +7486 ms
gap_task_to_1stspike_ms = first_spike_start_ms - eprime_end_ms   # +23486 ms

# Convert everything to seconds for plotting
def s(ms): return ms / 1000.0

print("=== TIMING ALIGNMENT SUMMARY ===")
print(f"E-Prime task start        : {s(eprime_start_ms):.3f} s  ({eprime_start_ms:.0f} ms)")
print(f"E-Prime task end          : {s(eprime_end_ms):.3f} s  ({eprime_end_ms:.0f} ms)")
print(f"Neural recording start    : {s(neural_start_ms):.3f} s  ({neural_start_ms:.0f} ms)")
print(f"First neural spike start  : {s(first_spike_start_ms):.3f} s  ({first_spike_start_ms:.0f} ms)")
print(f"First neural spike end    : {s(first_spike_end_ms):.3f} s  ({first_spike_end_ms:.0f} ms)")
print(f"Last  neural spike start  : {s(last_spike_start_ms):.3f} s  ({last_spike_start_ms:.0f} ms)")
print(f"Last  neural spike end    : {s(last_spike_end_ms):.3f} s  ({last_spike_end_ms:.0f} ms)")
print(f"Neural recording end      : {s(neural_end_ms):.3f} s  ({neural_end_ms:.0f} ms)")
print()
print(f"GAP: task end -> neural recording start : {gap_task_to_neural_ms:.0f} ms  = {gap_task_to_neural_ms/1000:.2f} s")
print(f"GAP: task end -> first neural spike     : {gap_task_to_1stspike_ms:.0f} ms = {gap_task_to_1stspike_ms/1000:.2f} s")
print()
print("CONCLUSION: Neural spikes are 100% post-task (DBS adaptive cycling).")
print("Task-locked spikes occurred DURING task (157-342s) but neural")
print("recording started AFTER task ended (at 349s).")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE: 3-panel timing alignment
# ═══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(22, 14), facecolor='white')
fig.suptitle(
    'E-Prime vs Neural Stimulation — Timing Alignment Analysis\n'
    'Subject 6  |  Session 2026-03-05  |  ATN-DBS (Medtronic Percept)',
    fontsize=14, fontweight='bold', y=0.98
)

gs = fig.add_gridspec(3, 1, height_ratios=[1.1, 1.6, 1.6],
                      hspace=0.55, left=0.08, right=0.95,
                      top=0.91, bottom=0.07)

ax_top = fig.add_subplot(gs[0])   # Full timeline ruler
ax_mid = fig.add_subplot(gs[1])   # LFP mA trace + spike markers
ax_bot = fig.add_subplot(gs[2])   # Zoomed LFP mA + E-Prime events

# ── COLOURS ──────────────────────────────────────────────────────────────────
C_TASK    = '#1565C0'   # dark blue  — E-Prime task window
C_NEURAL  = '#00897B'   # teal       — neural recording window
C_SPIKE   = '#E91E63'   # pink-red   — neural spikes
C_GAP     = '#FF6F00'   # amber      — the gap
C_FIX     = '#2196F3'   # blue       — fixation
C_STIM_EV = '#FF9800'   # orange     — stimulus digit
C_RESP    = '#9C27B0'   # purple     — response

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 1 — Full shared timeline ruler
# ═══════════════════════════════════════════════════════════════════════════════
ax_top.set_facecolor('#FAFAFA')

# Full timeline span
tl_start = 100
tl_end   = 680
ax_top.set_xlim(tl_start, tl_end)
ax_top.set_ylim(-0.5, 3.2)
ax_top.axis('off')
ax_top.set_title('Panel 1 — Full Shared Timeline (E-Prime clock, seconds)',
                 fontsize=11, fontweight='bold', loc='left', pad=4)

# Timeline axis line
ax_top.annotate('', xy=(tl_end-2, 0), xytext=(tl_start, 0),
                arrowprops=dict(arrowstyle='->', color='#37474F', lw=1.5))
for tick in range(100, 681, 50):
    ax_top.axvline(tick, color='#B0BEC5', lw=0.8, ymin=0.0, ymax=0.22, clip_on=False)
    ax_top.text(tick, -0.38, f'{tick}s', ha='center', va='top',
                fontsize=8, color='#546E7A')
ax_top.text(tl_end, -0.38, 'E-Prime time (s)', ha='right', va='top',
            fontsize=8.5, color='#37474F', style='italic')

# E-Prime task bar
ax_top.broken_barh([(s(eprime_start_ms), s(eprime_end_ms - eprime_start_ms))],
                   (1.5, 0.55), facecolors=C_TASK, alpha=0.85, edgecolor='white')
ax_top.text((s(eprime_start_ms) + s(eprime_end_ms)) / 2, 1.77,
            f'E-Prime Task\n{s(eprime_start_ms):.1f}s → {s(eprime_end_ms):.1f}s  '
            f'(dur: {s(eprime_end_ms-eprime_start_ms):.1f}s)',
            ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Neural recording bar
ax_top.broken_barh([(s(neural_start_ms), s(neural_end_ms - neural_start_ms))],
                   (0.5, 0.55), facecolors=C_NEURAL, alpha=0.85, edgecolor='white')
ax_top.text((s(neural_start_ms) + s(neural_end_ms)) / 2, 0.77,
            f'Neural Recording\n{s(neural_start_ms):.1f}s → {s(neural_end_ms):.1f}s  '
            f'(dur: {s(neural_end_ms-neural_start_ms):.1f}s)',
            ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# GAP annotation
gap_mid = (s(eprime_end_ms) + s(neural_start_ms)) / 2
ax_top.annotate('', xy=(s(neural_start_ms), 1.05), xytext=(s(eprime_end_ms), 1.05),
                arrowprops=dict(arrowstyle='<->', color=C_GAP, lw=2.0))
ax_top.text(gap_mid, 1.18,
            f'GAP\n{gap_task_to_neural_ms/1000:.1f}s',
            ha='center', va='bottom', fontsize=8.5, color=C_GAP, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', fc='#FFF9C4', ec=C_GAP, lw=1))

# First and last neural spikes
ax_top.axvline(s(first_spike_start_ms), color=C_SPIKE, lw=2.0, ls='--',
               ymin=0.13, ymax=0.58)
ax_top.text(s(first_spike_start_ms), 0.32, '1st\nspike',
            ha='center', va='center', fontsize=7.5, color=C_SPIKE, fontweight='bold')

ax_top.axvline(s(last_spike_end_ms), color=C_SPIKE, lw=2.0, ls='--',
               ymin=0.13, ymax=0.58)
ax_top.text(s(last_spike_end_ms), 0.32, 'last\nspike',
            ha='center', va='center', fontsize=7.5, color=C_SPIKE, fontweight='bold')

# Key vertical markers
for t_ms, label, col, ypos in [
    (eprime_start_ms, f'Task start\n{s(eprime_start_ms):.1f}s', C_TASK, 2.5),
    (eprime_end_ms,   f'Task end\n{s(eprime_end_ms):.1f}s',   C_TASK, 2.5),
    (neural_start_ms, f'Neural start\n{s(neural_start_ms):.1f}s', C_NEURAL, 2.5),
]:
    ax_top.axvline(s(t_ms), color=col, lw=1.5, ls=':', alpha=0.7,
                   ymin=0.0, ymax=0.95)
    ax_top.text(s(t_ms), ypos, label, ha='center', va='bottom',
                fontsize=7.5, color=col, fontweight='bold')

# Legend
handles_top = [
    mpatches.Patch(fc=C_TASK,   label='E-Prime task window'),
    mpatches.Patch(fc=C_NEURAL, label='Neural recording window'),
    Line2D([0],[0], color=C_SPIKE, lw=2, ls='--', label='Neural spike boundaries'),
    mpatches.Patch(fc='#FFF9C4', ec=C_GAP, label=f'Gap between task & neural ({gap_task_to_neural_ms/1000:.1f}s)'),
]
ax_top.legend(handles=handles_top, loc='upper right', fontsize=8,
              framealpha=0.9, ncol=2, bbox_to_anchor=(1.0, 1.55))

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 2 — Full LFP mA trace with all spikes labelled
# ═══════════════════════════════════════════════════════════════════════════════
ax_mid.set_facecolor('#FAFAFA')

lfp_t  = lfp_s['eprime_time_ms'].values / 1000
lfp_ma = lfp_s['stim_ma_left'].values

# Draw the mA trace
ax_mid.step(lfp_t, lfp_ma, where='post', color=C_NEURAL, lw=1.8, alpha=0.9,
            label='Stim amplitude Left ATN (mA)')
ax_mid.fill_between(lfp_t, lfp_ma, step='post', alpha=0.20, color=C_NEURAL)

# Shade each spike (0->mA->0 full cycle)
for i, (_, row_s) in enumerate(spike_starts.iterrows()):
    s_on  = row_s['eprime_time_ms'] / 1000
    # Match with corresponding end
    ends_after = spike_ends[spike_ends['eprime_time_ms'] > row_s['eprime_time_ms']]
    if len(ends_after) == 0:
        continue
    s_off = ends_after.iloc[0]['eprime_time_ms'] / 1000
    ax_mid.axvspan(s_on, s_off, color=C_SPIKE, alpha=0.18, zorder=0)
    # Label spike number above
    ax_mid.text((s_on + s_off)/2, lfp_ma.max()*0.97,
                f'S{i+1}', ha='center', va='top',
                fontsize=7, color=C_SPIKE, fontweight='bold')

# Mark task end
ax_mid.axvline(s(eprime_end_ms), color=C_TASK, lw=2.5, ls='--', alpha=0.9,
               label=f'E-Prime task end ({s(eprime_end_ms):.1f}s)')
ax_mid.text(s(eprime_end_ms) + 1, lfp_ma.max()*0.88,
            f'Task ended\n({s(eprime_end_ms):.1f}s)', color=C_TASK,
            fontsize=8.5, va='top', fontweight='bold')

# Mark neural start
ax_mid.axvline(s(neural_start_ms), color=C_NEURAL, lw=2.5, ls=':',
               alpha=0.9, label=f'Neural recording start ({s(neural_start_ms):.1f}s)')

# Mark first and last spike
ax_mid.annotate(f'First spike\n{s(first_spike_start_ms):.1f}s',
                xy=(s(first_spike_start_ms), 0.1),
                xytext=(s(first_spike_start_ms)+12, 0.35),
                fontsize=8.5, color=C_SPIKE, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C_SPIKE, lw=1.5))
ax_mid.annotate(f'Last spike\n{s(last_spike_start_ms):.1f}s',
                xy=(s(last_spike_start_ms), 0.1),
                xytext=(s(last_spike_start_ms)-40, 0.35),
                fontsize=8.5, color=C_SPIKE, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C_SPIKE, lw=1.5))

ax_mid.set_xlim(s(eprime_end_ms) - 5, s(neural_end_ms) + 5)
ax_mid.set_ylim(-0.02, lfp_ma.max() * 1.12)
ax_mid.set_ylabel('Stimulation amplitude (mA)', fontsize=10, fontweight='bold')
ax_mid.set_xlabel('E-Prime Time (s)', fontsize=10)
ax_mid.set_title(
    f'Panel 2 — Neural LFP Stimulation Amplitude (2 Hz)  |  '
    f'15 spikes detected (all post-task)  |  '
    f'First spike: {s(first_spike_start_ms):.1f}s → Last spike end: {s(last_spike_end_ms):.1f}s',
    fontsize=10, fontweight='bold', loc='left', pad=4)
ax_mid.legend(fontsize=8.5, loc='upper right', framealpha=0.9)
ax_mid.grid(axis='y', alpha=0.25, lw=0.7)
for spine in ['top', 'right']:
    ax_mid.spines[spine].set_visible(False)

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 3 — Zoomed: first spike + last spike side by side, with E-Prime events
# ═══════════════════════════════════════════════════════════════════════════════
ax_bot.set_facecolor('#FAFAFA')

# We will show a zoomed view of first spike region (left) and last spike (right)
# Split axes using a broken-axis style with a visual break
zoom_window = 30   # ±15s around each spike

# Plot full mA trace again (same as mid)
ax_bot.step(lfp_t, lfp_ma, where='post', color=C_NEURAL, lw=1.8, alpha=0.9)
ax_bot.fill_between(lfp_t, lfp_ma, step='post', alpha=0.15, color=C_NEURAL)

# Shade all spikes
for _, row_s in spike_starts.iterrows():
    s_on  = row_s['eprime_time_ms'] / 1000
    ends_after = spike_ends[spike_ends['eprime_time_ms'] > row_s['eprime_time_ms']]
    if len(ends_after) == 0:
        continue
    s_off = ends_after.iloc[0]['eprime_time_ms'] / 1000
    ax_bot.axvspan(s_on, s_off, color=C_SPIKE, alpha=0.22, zorder=0)

# E-Prime task band (even though task ended before recording, show its boundary)
ax_bot.axvline(s(eprime_end_ms), color=C_TASK, lw=2.5, ls='--', alpha=0.9)
ax_bot.text(s(eprime_end_ms) + 0.3, lfp_ma.max()*0.92,
            f'Task end\n({s(eprime_end_ms):.1f}s)',
            color=C_TASK, fontsize=8, va='top', fontweight='bold')
ax_bot.axvline(s(neural_start_ms), color='#00897B', lw=2, ls=':', alpha=0.7)
ax_bot.text(s(neural_start_ms) + 0.3, lfp_ma.max()*0.75,
            f'Neural start\n({s(neural_start_ms):.1f}s)',
            color='#00897B', fontsize=8, va='top', fontweight='bold')

# Annotate gap between task end and first spike
ax_bot.annotate('', xy=(s(first_spike_start_ms), lfp_ma.max()*0.55),
                xytext=(s(eprime_end_ms), lfp_ma.max()*0.55),
                arrowprops=dict(arrowstyle='<->', color=C_GAP, lw=2.0))
gap_label_x = (s(eprime_end_ms) + s(first_spike_start_ms)) / 2
ax_bot.text(gap_label_x, lfp_ma.max()*0.58,
            f'+{gap_task_to_1stspike_ms/1000:.1f}s from task end\nto first neural spike',
            ha='center', va='bottom', fontsize=8, color=C_GAP, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.25', fc='#FFF9C4', ec=C_GAP, lw=1))

# First spike detail box
ax_bot.annotate(
    f' First spike (S1)\n onset:  {s(first_spike_start_ms):.1f}s\n'
    f' offset: {s(first_spike_end_ms):.1f}s\n'
    f' dur:    {(first_spike_end_ms-first_spike_start_ms)/1000:.1f}s',
    xy=(s(first_spike_start_ms + (first_spike_end_ms-first_spike_start_ms)/2), 0.1),
    xytext=(s(first_spike_start_ms) + 8, 0.38),
    fontsize=8, color=C_SPIKE, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=C_SPIKE, lw=1.5, alpha=0.95),
    arrowprops=dict(arrowstyle='->', color=C_SPIKE, lw=1.5)
)

# Last spike detail box
ax_bot.annotate(
    f' Last spike (S15)\n onset:  {s(last_spike_start_ms):.1f}s\n'
    f' offset: {s(last_spike_end_ms):.1f}s\n'
    f' dur:    {(last_spike_end_ms-last_spike_start_ms)/1000:.1f}s',
    xy=(s(last_spike_start_ms + (last_spike_end_ms-last_spike_start_ms)/2), 0.1),
    xytext=(s(last_spike_start_ms) - 50, 0.38),
    fontsize=8, color=C_SPIKE, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=C_SPIKE, lw=1.5, alpha=0.95),
    arrowprops=dict(arrowstyle='->', color=C_SPIKE, lw=1.5)
)

# Big text verdict box
verdict_x = (s(eprime_end_ms) + s(neural_end_ms)) / 2
ax_bot.text(
    s(neural_start_ms) + 90, lfp_ma.max() * 0.60,
    '⚠  DO NOT MATCH\n'
    f'E-Prime task:   {s(eprime_start_ms):.1f}s – {s(eprime_end_ms):.1f}s\n'
    f'1st neural spike: {s(first_spike_start_ms):.1f}s  (+{gap_task_to_1stspike_ms/1000:.1f}s after task end)\n'
    f'Last spike end:   {s(last_spike_end_ms):.1f}s\n'
    f'Neural spikes = post-task DBS cycling, NOT task-locked ATN stimulation',
    ha='center', va='center', fontsize=9, color='#B71C1C',
    fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.6', fc='#FFEBEE', ec='#E53935', lw=2.0, alpha=0.97)
)

ax_bot.set_xlim(s(eprime_end_ms) - 8, s(neural_end_ms) + 5)
ax_bot.set_ylim(-0.02, lfp_ma.max() * 1.18)
ax_bot.set_ylabel('Stimulation amplitude (mA)', fontsize=10, fontweight='bold')
ax_bot.set_xlabel('E-Prime Time (s)', fontsize=10)
ax_bot.set_title(
    'Panel 3 — Zoomed: Task End Boundary + All Neural Spikes  '
    '(confirms zero overlap between task events and neural spikes)',
    fontsize=10, fontweight='bold', loc='left', pad=4)
ax_bot.grid(axis='y', alpha=0.25, lw=0.7)
for spine in ['top', 'right']:
    ax_bot.spines[spine].set_visible(False)

# ── Summary text box at bottom of figure ──────────────────────────────────────
summary = (
    f"E-Prime task:  {s(eprime_start_ms):.3f}s → {s(eprime_end_ms):.3f}s  "
    f"(duration {s(eprime_end_ms-eprime_start_ms):.1f}s)          "
    f"Neural recording:  {s(neural_start_ms):.3f}s → {s(neural_end_ms):.3f}s          "
    f"1st neural spike:  {s(first_spike_start_ms):.3f}s → {s(first_spike_end_ms):.3f}s          "
    f"Last neural spike:  {s(last_spike_start_ms):.3f}s → {s(last_spike_end_ms):.3f}s          "
    f"Gap (task end → neural start): {gap_task_to_neural_ms/1000:.2f}s          "
    f"Gap (task end → 1st spike): {gap_task_to_1stspike_ms/1000:.2f}s"
)
fig.text(0.5, 0.01, summary, ha='center', va='bottom', fontsize=8,
         color='#37474F', style='italic',
         bbox=dict(boxstyle='round,pad=0.4', fc='#ECEFF1', ec='#B0BEC5', lw=1))

out = os.path.join(PLOTS_DIR, 'timing_alignment.png')
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n✓ Saved: {out}")
print("Done.")

=== TIMING ALIGNMENT SUMMARY ===
E-Prime task start        : 157.381 s  (157381 ms)
E-Prime task end          : 342.014 s  (342014 ms)
Neural recording start    : 349.000 s  (349000 ms)
First neural spike start  : 365.500 s  (365500 ms)
First neural spike end    : 368.000 s  (368000 ms)
Last  neural spike start  : 622.000 s  (622000 ms)
Last  neural spike end    : 624.000 s  (624000 ms)
Neural recording end      : 628.500 s  (628500 ms)

GAP: task end -> neural recording start : 6986 ms  = 6.99 s
GAP: task end -> first neural spike     : 23486 ms = 23.49 s

CONCLUSION: Neural spikes are 100% post-task (DBS adaptive cycling).
Task-locked spikes occurred DURING task (157-342s) but neural
recording started AFTER task ended (at 349s).

✓ Saved: C:\Users\ASSUS\ATN\Digit Span Backwards\Plots\timing_alignment.png
Done.


In [3]:
"""
Script 6: Overlaid timeline — warped neural spikes onto E-Prime events.

METHOD:
  Linear time-warping anchored at two points:
    anchor 1: first spike onset  (365,500 ms neural)  -> task start (157,381 ms eprime)
    anchor 2: last  spike onset  (622,000 ms neural)  -> task end   (342,014 ms eprime)

  Formula: warped_ms = ep_start + (neural_ms - first_spike) * scale
           scale     = (ep_end - ep_start) / (last_spike - first_spike) = 0.7198

  After warping: every spike and every sample of the 250Hz LFP is expressed
  in the same millisecond units as the E-Prime events, so they can be plotted
  on a shared axis and compared.

NOTE:
  This is a forensic alignment, not a ground-truth sync. The task and neural
  recording did not overlap. The warping is the best we can do with this data.
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import os, warnings
warnings.filterwarnings('ignore')

PLOTS_DIR = "C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── load ──────────────────────────────────────────────────────────────────────
events = pd.read_csv('C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data\\events_data_processed.csv')
lfp    = pd.read_csv('C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data\\neural_lfp_power_processed.csv')
neural = pd.read_csv('C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Preprocessed Data\\neural_data_processed.csv')

left  = neural[neural['hemisphere']=='Left'].sort_values('eprime_time_ms').reset_index(drop=True)
right = neural[neural['hemisphere']=='Right'].sort_values('eprime_time_ms').reset_index(drop=True)

# ── spikes ────────────────────────────────────────────────────────────────────
lfp_s = lfp.sort_values('eprime_time_ms').reset_index(drop=True)
lfp_s['prev_ma'] = lfp_s['stim_ma_left'].shift(1).fillna(0)
spike_on  = lfp_s[(lfp_s['prev_ma']==0) & (lfp_s['stim_ma_left']>0)].copy()
spike_off = lfp_s[(lfp_s['prev_ma']>0)  & (lfp_s['stim_ma_left']==0)].copy()
spike_onsets  = spike_on['eprime_time_ms'].values
spike_offsets = spike_off['eprime_time_ms'].values

# ── warping ───────────────────────────────────────────────────────────────────
ep_start    = events['fixation_onset_ms'].min()
ep_end      = events['feedback_offset_ms'].max()
first_spike = spike_onsets[0]
last_spike  = spike_onsets[-1]
scale       = (ep_end - ep_start) / (last_spike - first_spike)

def warp(t):
    return ep_start + (t - first_spike) * scale

warped_onsets  = np.array([warp(t) for t in spike_onsets])
warped_offsets = np.array([warp(t) for t in spike_offsets])

# Warp LFP signal (downsample for speed)
DS = 5
left_t_warped  = warp(left['eprime_time_ms'].values[::DS])
left_v         = left['amplitude_uv'].values[::DS]
right_t_warped = warp(right['eprime_time_ms'].values[::DS])
right_v        = right['amplitude_uv'].values[::DS]

# Warp full LFP power
lfp_t_warped = warp(lfp_s['eprime_time_ms'].values)
lfp_ma       = lfp_s['stim_ma_left'].values

# ── nearest event per spike ───────────────────────────────────────────────────
all_ev = []
for _, row in events.iterrows():
    all_ev += [
        {'time': row['fixation_onset_ms'],  'type': 'Fixation',
         'label': f"Fix B{int(row.block)}T{int(row.trial)}S{int(row.sub_trial)} digit={int(row.digit_shown)}",
         'color': '#2196F3', 'span': int(row.span_size), 'acc': int(row.is_correct)},
        {'time': row['stimulus_onset_ms'],  'type': 'Stimulus',
         'label': f'Stim B{int(row.block)}T{int(row.trial)}S{int(row.sub_trial)} "{int(row.digit_shown)}"',
         'color': '#FF9800', 'span': int(row.span_size), 'acc': int(row.is_correct)},
    ]
    if int(row.sub_trial) == int(row.span_size):
        acc = int(row.is_correct)
        all_ev += [
            {'time': row['response_onset_ms'], 'type': 'Response',
             'label': f"Resp B{int(row.block)}T{int(row.trial)}", 'color': '#9C27B0',
             'span': int(row.span_size), 'acc': acc},
            {'time': row['feedback_onset_ms'], 'type': 'Feedback',
             'label': f"Feed B{int(row.block)}T{int(row.trial)} {'✓' if acc else '✗'}",
             'color': '#4CAF50' if acc else '#F44336',
             'span': int(row.span_size), 'acc': acc},
        ]
df_ev = pd.DataFrame(all_ev).sort_values('time').reset_index(drop=True)

# Find nearest event for each spike
spike_nearest = []
for i, w in enumerate(warped_onsets):
    diffs = np.abs(df_ev['time'].values - w)
    ni    = np.argmin(diffs)
    row   = df_ev.iloc[ni]
    spike_nearest.append({
        'spike_i'    : i+1,
        'warped_ms'  : w,
        'neural_ms'  : spike_onsets[i],
        'ev_type'    : row['type'],
        'ev_label'   : row['label'],
        'ev_time'    : row['time'],
        'ev_color'   : row['color'],
        'diff_ms'    : w - row['time'],
        'span'       : row['span'],
        'acc'        : row['acc'],
    })
df_sn = pd.DataFrame(spike_nearest)

print("=== SPIKE TO EVENT MAPPING ===")
print(df_sn[['spike_i','warped_ms','ev_type','ev_label','diff_ms']].to_string())
print()
print(f"Scale: {scale:.6f}  |  anchor 1: neural {first_spike:.0f}ms -> eprime {ep_start:.0f}ms")
print(f"                     |  anchor 2: neural {last_spike:.0f}ms  -> eprime {ep_end:.0f}ms")

# ── colours ───────────────────────────────────────────────────────────────────
C = dict(fix='#2196F3', stim='#FF9800', resp='#9C27B0',
         feed_ok='#4CAF50', feed_er='#F44336', spike='#E91E63',
         left='#1565C0', right='#0288D1', lfp='#00897B', shade='#ECEFF1')
SPAN_C = {2:'#7B1FA2', 3:'#0277BD', 4:'#2E7D32', 5:'#BF360C'}

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE: 4 panels
# ═══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(28, 22), facecolor='white')
fig.suptitle(
    'Warped Overlay: Neural Spikes Aligned to E-Prime Task Events\n'
    'Linear time-warp:  Spike 1 → Task start  |  Spike 15 → Task end  |  '
    f'Scale = {scale:.4f}  |  Subject 6, 2026-03-05',
    fontsize=14, fontweight='bold', y=0.995
)

gs = GridSpec(4, 1, figure=fig,
              height_ratios=[1.2, 1.8, 1.6, 1.4],
              hspace=0.50, left=0.06, right=0.97, top=0.96, bottom=0.04)

ax0 = fig.add_subplot(gs[0])   # stimulation spikes + event lines
ax1 = fig.add_subplot(gs[1])   # 250Hz LFP both channels (warped)
ax2 = fig.add_subplot(gs[2])   # LFP power + mA (warped)
ax3 = fig.add_subplot(gs[3])   # event detail (spike labels vs event labels)

x_lo = ep_start / 1000 - 1
x_hi = ep_end   / 1000 + 1

# ── helper: shade by span ────────────────────────────────────────────────────
def shade_spans(ax, ybot, ytop):
    for (blk, trl), grp in events.groupby(['block','trial']):
        t0 = grp['fixation_onset_ms'].min() / 1000
        t1 = grp['feedback_offset_ms'].max() / 1000
        span = int(grp['span_size'].iloc[0])
        ax.axvspan(t0, t1, color=SPAN_C[span], alpha=0.06, zorder=0)

def draw_event_lines(ax, alpha=0.40, lw=0.9):
    for _, row in events.iterrows():
        sub  = int(row['sub_trial'])
        span = int(row['span_size'])
        acc  = int(row['is_correct'])
        ax.axvline(row['fixation_onset_ms']/1000,  color=C['fix'],  lw=lw, ls='--', alpha=alpha)
        ax.axvline(row['stimulus_onset_ms']/1000,  color=C['stim'], lw=lw, ls='-',  alpha=alpha)
        if sub == span:
            ax.axvline(row['response_onset_ms']/1000, color=C['resp'], lw=lw*1.4, ls='-.', alpha=alpha+.15)
            fc = C['feed_ok'] if acc else C['feed_er']
            ax.axvline(row['feedback_onset_ms']/1000, color=fc, lw=lw*1.4, ls=':', alpha=alpha+.15)

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 0 — Spike train overlaid on event lines
# ═══════════════════════════════════════════════════════════════════════════════
ax0.set_facecolor('white')
shade_spans(ax0, 0, 1)
draw_event_lines(ax0, alpha=0.30, lw=0.8)

# Draw warped spikes as filled stems
for i, (w_on, w_off) in enumerate(zip(warped_onsets, warped_offsets)):
    t_on  = w_on  / 1000
    t_off = w_off / 1000
    ec    = df_sn.iloc[i]['ev_color']
    # Stem
    ax0.annotate('', xy=(t_on, 0.85), xytext=(t_on, 0.05),
                 arrowprops=dict(arrowstyle='->', color=C['spike'], lw=2.2))
    # Horizontal bar showing spike duration
    ax0.plot([t_on, t_off], [0.88, 0.88], color=C['spike'], lw=3, solid_capstyle='round', alpha=0.7)
    # Spike number
    ax0.text(t_on, 0.91, f'S{i+1}', ha='center', va='bottom',
             fontsize=6.5, color=C['spike'], fontweight='bold')

# Block labels
for blk, grp in events.groupby('block'):
    mid  = (grp['fixation_onset_ms'].min() + grp['feedback_offset_ms'].max()) / 2 / 1000
    span = int(grp['span_size'].iloc[0])
    ax0.text(mid, 0.01, f'Blk{blk}\n{span}d', ha='center', va='bottom',
             fontsize=7.5, color=SPAN_C[span], fontweight='bold')

ax0.set_xlim(x_lo, x_hi)
ax0.set_ylim(0, 1.0)
ax0.set_yticks([0.05, 0.85])
ax0.set_yticklabels(['0 mA', '0.1 mA'], fontsize=9)
ax0.set_ylabel('Stim (mA)', fontsize=10, fontweight='bold')
ax0.set_title(
    'Panel 1 — Warped neural stimulation spikes (S1–S15) overlaid on E-Prime event markers\n'
    'Spike duration shown as horizontal bar at top.  Spike arrows coloured by nearest matched event.',
    fontsize=10, loc='left', pad=3)
for sp in ['top','right']: ax0.spines[sp].set_visible(False)

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 1 — 250 Hz LFP both channels (warped)
# ═══════════════════════════════════════════════════════════════════════════════
ax1.set_facecolor('#FAFAFA')
shade_spans(ax1, 0, 1)
draw_event_lines(ax1, alpha=0.18, lw=0.7)

ax1.plot(left_t_warped/1000,  left_v,  color=C['left'],  lw=0.5, alpha=0.85, label='Left ATN (ZERO_THREE_LEFT)')
ax1.plot(right_t_warped/1000, right_v, color=C['right'], lw=0.5, alpha=0.75, label='Right ATN (ZERO_THREE_RIGHT)')

# Mark each warped spike onset on LFP
for i, w in enumerate(warped_onsets):
    ax1.axvline(w/1000, color=C['spike'], lw=1.8, ls='-', alpha=0.85)
    ax1.text(w/1000, ax1.get_ylim()[1] if False else 80, f'S{i+1}',
             ha='center', va='top', fontsize=6, color=C['spike'], fontweight='bold')

ax1.set_xlim(x_lo, x_hi)
ax1.set_ylabel('LFP Amplitude (µV)', fontsize=10, fontweight='bold')
ax1.set_title(
    'Panel 2 — Warped 250 Hz LFP signal, Left & Right ATN  '
    '(time-warped so spike 1 = task start, spike 15 = task end)',
    fontsize=10, loc='left', pad=3)
ax1.legend(fontsize=9, loc='upper right', framealpha=0.9)
ax1.grid(axis='y', alpha=0.25, lw=0.6)
for sp in ['top','right']: ax1.spines[sp].set_visible(False)
# Add spike labels at top after axis is set
ymax_lfp = max(left_v.max(), right_v.max())
for i, w in enumerate(warped_onsets):
    ax1.text(w/1000, ymax_lfp * 0.95, f'S{i+1}',
             ha='center', va='top', fontsize=6.5, color=C['spike'], fontweight='bold')

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 2 — LFP power + mA (warped)
# ═══════════════════════════════════════════════════════════════════════════════
ax2.set_facecolor('#FAFAFA')
ax2b = ax2.twinx()
shade_spans(ax2, 0, 1)
draw_event_lines(ax2, alpha=0.18, lw=0.7)

ax2.fill_between(lfp_t_warped/1000, lfp_s['lfp_power_left'].values,
                 alpha=0.25, color=C['lfp'])
ax2.plot(lfp_t_warped/1000, lfp_s['lfp_power_left'].values,
         color=C['lfp'], lw=1.5, label='LFP power Left (2Hz, warped)')

ax2b.step(lfp_t_warped/1000, lfp_ma, where='post',
          color=C['spike'], lw=2.0, alpha=0.85, label='Stim mA Left')
ax2b.set_ylabel('Stim mA', fontsize=9, color=C['spike'])
ax2b.tick_params(axis='y', colors=C['spike'])
ax2b.set_ylim(-0.05, lfp_ma.max() * 1.3)

for i, w in enumerate(warped_onsets):
    ax2.axvline(w/1000, color=C['spike'], lw=1.5, ls='-', alpha=0.7)

ax2.set_xlim(x_lo, x_hi)
ax2.set_ylabel('LFP Power (device units)', fontsize=10, fontweight='bold')
ax2.set_title(
    'Panel 3 — LFP power envelope (2Hz, warped) + stimulation amplitude (mA)',
    fontsize=10, loc='left', pad=3)
lines1, labs1 = ax2.get_legend_handles_labels()
lines2, labs2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, labs1+labs2, fontsize=8.5, loc='upper right', framealpha=0.9)
ax2.grid(axis='y', alpha=0.25, lw=0.6)
for sp in ['top','right']: ax2.spines[sp].set_visible(False)

# ═══════════════════════════════════════════════════════════════════════════════
# PANEL 3 — Spike-to-event matching table
# ═══════════════════════════════════════════════════════════════════════════════
ax3.set_facecolor('white')
ax3.axis('off')
ax3.set_title(
    'Panel 4 — Each warped spike matched to its nearest E-Prime event',
    fontsize=10, loc='left', pad=3)

# Draw mini timeline + labels
y_spine = 0.55
ax3.axhline(y_spine, xmin=0.01, xmax=0.99, color='#B0BEC5', lw=1.0)

# Normalise x to [0.02, 0.98]
def norm_x(ms):
    return 0.02 + (ms - ep_start) / (ep_end - ep_start) * 0.96

for _, row in df_sn.iterrows():
    nx   = norm_x(row['warped_ms'])
    ec   = row['ev_color']
    si   = int(row['spike_i'])
    # Spike tick
    ax3.plot([nx, nx], [y_spine-0.04, y_spine+0.04], color=C['spike'], lw=2.0)
    ax3.plot(nx, y_spine, 'o', color=C['spike'], ms=5, zorder=5)
    # Event label alternating above/below
    ya = y_spine + (0.32 if si % 2 == 1 else -0.12)
    evt = row['ev_type'][:4]
    blk_trl = row['ev_label'].split(' ')[1] if ' ' in row['ev_label'] else ''
    diff_s  = row['diff_ms'] / 1000
    lbl     = f"S{si}\n{evt}\n{blk_trl}\nΔ{diff_s:+.1f}s"
    ax3.text(nx, ya, lbl, ha='center', va='center',
             fontsize=5.8, color=ec, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=ec, lw=0.8, alpha=0.92))
    # Leader line from box to tick
    ax3.plot([nx, nx], [y_spine+(0.06 if si%2==1 else -0.06),
                         ya - (0.10 if si%2==1 else -0.10)],
             color=ec, lw=0.7, ls='--', alpha=0.6)

# Block shading along mini timeline
for (blk, trl), grp in events.groupby(['block','trial']):
    t0 = norm_x(grp['fixation_onset_ms'].min())
    t1 = norm_x(grp['feedback_offset_ms'].max())
    span = int(grp['span_size'].iloc[0])
    ax3.axhspan(y_spine-0.04, y_spine+0.04,
                xmin=t0, xmax=t1, color=SPAN_C[span], alpha=0.18, zorder=0)

# Anchor labels
ax3.text(norm_x(ep_start), y_spine-0.20, f'Task start\n{ep_start/1000:.1f}s\n= Spike 1',
         ha='center', va='top', fontsize=8, color=C['spike'], fontweight='bold')
ax3.text(norm_x(ep_end),   y_spine-0.20, f'Task end\n{ep_end/1000:.1f}s\n= Spike 15',
         ha='center', va='top', fontsize=8, color=C['spike'], fontweight='bold')

# Span legend
for i, (span, col) in enumerate(SPAN_C.items()):
    ax3.add_patch(mpatches.FancyBboxPatch((0.01+i*0.10, 0.03), 0.08, 0.10,
                  boxstyle='round,pad=0.01', fc=col, ec=col, alpha=0.25))
    ax3.text(0.05+i*0.10, 0.08, f'{span}-digit', ha='center', va='center',
             fontsize=8, color=col, fontweight='bold')

ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)

# ── shared x setup ─────────────────────────────────────────────────────────────
for ax in [ax0, ax1, ax2]:
    ax.set_xlabel('E-Prime time (s)  [neural axis linearly warped to match]', fontsize=10)
    ax.tick_params(axis='both', labelsize=9)

# ── global legend ─────────────────────────────────────────────────────────────
legend_elems = [
    Line2D([0],[0], color=C['fix'],     lw=1.5, ls='--', label='Fixation onset'),
    Line2D([0],[0], color=C['stim'],    lw=1.5, ls='-',  label='Stimulus onset'),
    Line2D([0],[0], color=C['resp'],    lw=1.5, ls='-.', label='Response onset'),
    Line2D([0],[0], color=C['feed_ok'],lw=1.5, ls=':',  label='Feedback (correct)'),
    Line2D([0],[0], color=C['feed_er'],lw=1.5, ls=':',  label='Feedback (incorrect)'),
    mpatches.Patch(fc=C['spike'], alpha=0.8, label='Warped neural spike (0.1 mA)'),
    Line2D([0],[0], color=C['left'],   lw=1.5, label='LFP Left ATN (250Hz)'),
    Line2D([0],[0], color=C['right'],  lw=1.5, label='LFP Right ATN (250Hz)'),
]
fig.legend(handles=legend_elems, loc='lower center', ncol=8, fontsize=9,
           framealpha=0.95, bbox_to_anchor=(0.5, 0.005))

fig.text(0.5, 0.001,
    '⚠  Forensic alignment only. Time-warp assumes linear correspondence between '
    'neural spike spacing and E-Prime event spacing. '
    'True task-locked recording was not captured in this session.',
    ha='center', va='bottom', fontsize=8, color='#607D8B', style='italic')

out = os.path.join(PLOTS_DIR, 'overlay_alignment.png')
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n✓ Saved: {out}")

=== SPIKE TO EVENT MAPPING ===
    spike_i      warped_ms   ev_type            ev_label      diff_ms
0         1  157381.000000  Fixation  Fix B1T1S1 digit=4     0.000000
1         2  205248.814815  Fixation  Fix B3T2S1 digit=4  -640.185185
2         3  270752.140351  Stimulus     Stim B5T2S4 "5"    71.140351
3         4  273271.499025  Response           Resp B5T2  1574.499025
4         5  307822.703704  Fixation  Fix B7T1S1 digit=4   -99.296296
5         6  312501.512671  Stimulus     Stim B7T1S3 "5"  -466.487329
6         7  315380.779727  Stimulus     Stim B7T1S4 "2"   391.779727
7         8  318260.046784  Response           Resp B7T1   225.046784
8         9  321139.313840  Feedback         Feed B7T1 ✗ -2324.686160
9        10  323298.764133  Feedback         Feed B7T1 ✗  -165.235867
10       11  325098.306043  Fixation  Fix B7T2S1 digit=5    84.306043
11       12  330496.931774  Stimulus     Stim B7T2S3 "3"   432.931774
12       13  333376.198830  Fixation  Fix B7T2S5 digit=4   